# Guided walkthrough: the "Add to cart" button test

Read the main `README.md` first. It has the scenario, the facts you can rely on, and the data dictionary.

Run the cells in order. The notebook asks you to make decisions along the way. Write them in `plan_template.md` or in the markdown cells.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from stats import proportion_users_needed_per_arm, continuous_users_needed_per_arm, two_proportion_test, welch_test, students_t_test, mann_whitney_test

DATA = Path("../data")

## Step 1: Plan before you look at any data

Decide, in writing, how you will judge this test.

The decisions you need to make:

1. What is your **metric**, and what is one row of data for it?
2. Which **test** will you use?
3. What is the **smallest difference** between the buttons that would justify rolling out the redesign?
4. What **significance level** and **power** will you use?
5. How many users do you need, and **how many days** will that take?
6. What will you do for each possible result?

The helper below answers question 5 for a click-rate comparison. It tells you how many users each group needs to detect a given difference with the chosen significance level and power.

In [ ]:
baseline = 0.10           # from the README: about 10% of users clicked the current button
new_users_per_day = 800   # from the README
alpha = 100               # false positive probability
power = 100               # false negative probability

lift = 0.90               # the difference you want to detect. Is this relative or absolute?


n = proportion_users_needed_per_arm(baseline, lift, alpha, power) 
print(f"Users needed per group: {round(n):,}")
print(f"Days needed at {new_users_per_day} new users per day: {round(2 * n / new_users_per_day, 1)}")

Is the answer reasonable? How would you know? Change `lift` to other values and compare the results, then decide which difference you will plan for and why.

Which of these differences can you detect within the two-week limit? Would you recommend a shorter or longer experiment?

Try swapping out `proportion_users_needed_per_arm` for `continuous_users_needed_per_arm`. Does n change? By how much? Why?

**Fill in `plan_template.md` now, before Step 2.**

## Step 2: Load the first five days

In [ ]:
early = pd.read_csv(DATA / "experiment_days_1-5.csv", parse_dates=["timestamp"])
early.head()

In [ ]:
print("Rows (visits):", len(early))
print("Distinct users:", early["user_id"].nunique())
early.groupby("button")["user_id"].nunique()

## Step 3: One row per user

The data has one row per **visit**, but the people who were randomly assigned a button are **users**. A user who visits four times is one person, not four independent observations.

Collapse the visits so that each user is a single row: which button they saw, and whether they clicked at least once.

In [ ]:
def per_user(visits):
    return visits.groupby("user_id").agg(
        button=("button", "first"),
        clicked=("clicked", "max"),
        visits=("visit_id", "count"),
    )


users = per_user(early)
users.head()

In [ ]:
summary = users.groupby("button").agg(users=("clicked", "size"), clicked=("clicked", "sum"))
summary["click_rate"] = summary["clicked"] / summary["users"]
summary

In [ ]:
# # Here's what the data would look like if you didn't aggregate by user. What differences do you seen? 
# # How would this mistake influence your decision?

# visit_summary = early.groupby("button").agg(
#     visits=("clicked", "size"),
#     clicks=("clicked", "sum"),
#     users=("user_id", "nunique"),
# )
# visit_summary["click_rate_per_visit"] = visit_summary["clicks"] / visit_summary["visits"]
# display(visit_summary)

# def test_visits(visits, test):
#     a = visits[visits["button"] == "A"]
#     b = visits[visits["button"] == "B"]
#     return test(a["clicked"].sum(), len(a), b["clicked"].sum(), len(b))


# pd.Series(test_visits(early, test)).round(4)

## Step 4: Test the difference

Now it is the two-group comparison from stats: two groups of users, each either clicked or did not.

- Null hypothesis: the click rate is the same for A and B.
- Alternative hypothesis: the click rates differ.

Choose one of the 4 tests from stats.py

In [ ]:
def test_users(user_table, test):
    a = user_table[user_table["button"] == "A"]
    b = user_table[user_table["button"] == "B"]
    return test(a["clicked"].sum(), len(a), b["clicked"].sum(), len(b))


pd.Series(test_users(users, *test_here*)).round(4) # choose a test from the stats.py file for test_here

## Step 5: What do you tell the product manager today?

Your boss is asking you if we can conclude the experiment early.

Write down:
- What the result is, in plain language.
- What you recommend: roll out A, keep B, or keep running the test.

## Step 6: All 14 days

Load the rest of the data.

Users who first appeared in days 1 to 5 and came back later show up in both files. Grouping by `user_id` handles that for you.

In [ ]:
SET = "set1"   # try "set2" later

later = pd.read_csv(DATA / f"experiment_days_6-14_{SET}.csv", parse_dates=["timestamp"])
full = pd.concat([early, later], ignore_index=True)
full_users = per_user(full)

print("Distinct users:", len(full_users))
pd.Series(test_users(full_users, *test_here*)).round(4) # switch out for the other statistical tests. Do the results change? Why do you think that is?

In [ ]:
# # Again, what would the conclusion look like if you didn't aggregate by user?
# display(test_visits(fullm *test_here*)) # choose a test from the stats.py file for test_here

## Step 7: Look back, day by day

The table below shows what the analysis would have said at the end of each day, using only the data available up to that day.

In [ ]:
start = pd.Timestamp("2026-03-02")
full["day"] = (full["timestamp"] - start).dt.days + 1

rows = []
for d in range(1, 15):
    seen = per_user(full[full["day"] <= d])
    res = test_users(seen, *test_here*)
    rows.append({
        "through day": d, "users": len(seen),
        "rate A": res["rate_A"], "rate B": res["rate_B"], "p-value": res["p_value"],
    })
pd.DataFrame(rows).round(4)

## Reflect

- How did your day-5 recommendation compare with what you see after 14 days?
- How did your significance level change what you would have done on day 5?
- Switch `SET` in Step 6 to the other file and run from there. What changes, and what does that tell you about what day 5 could and could not show?